In [1]:
import pandas as pd

df = pd.DataFrame({
    "Region": ["North","North","South","South","East","East","West","West"],
    "Product": ["A","B","A","C","B","C","A","B"],
    "Sales":   [100,200,300,400,150,250,500,600],
    "Discount":[5,10,15,20,5,10,25,30]
})

print(df)

  Region Product  Sales  Discount
0  North       A    100         5
1  North       B    200        10
2  South       A    300        15
3  South       C    400        20
4   East       B    150         5
5   East       C    250        10
6   West       A    500        25
7   West       B    600        30


1. Transform

In [2]:
# Normalize sales within each region(divide each sale by the region's mean)
df["normalized_sales"]=df.groupby("Region")["Sales"].transform(lambda x: x/x.mean())
print(df)

  Region Product  Sales  Discount  normalized_sales
0  North       A    100         5          0.666667
1  North       B    200        10          1.333333
2  South       A    300        15          0.857143
3  South       C    400        20          1.142857
4   East       B    150         5          0.750000
5   East       C    250        10          1.250000
6   West       A    500        25          0.909091
7   West       B    600        30          1.090909


In [ ]:
# Compute the difference between each discount and the group's max discount.
df["Discount_Diff"]=df.groupby("Region")["Discount"].transform(lambda x: x.max()-x)
print(df)

  Region Product  Sales  Discount  normalized_sales  Discount_Diff
0  North       A    100         5          0.666667              5
1  North       B    200        10          1.333333              0
2  South       A    300        15          0.857143              5
3  South       C    400        20          1.142857              0
4   East       B    150         5          0.750000              5
5   East       C    250        10          1.250000              0
6   West       A    500        25          0.909091              5
7   West       B    600        30          1.090909              0


# Filter

In [6]:
# Keep only regions where total sales exceed 700.
filtered_regions=df.groupby("Region").filter(lambda x:x["Sales"].sum()>700)
print(filtered_regions)

  Region Product  Sales  Discount  normalized_sales  Discount_Diff
6   West       A    500        25          0.909091              5
7   West       B    600        30          1.090909              0


In [13]:
# Keep only products that appear more than once across all regions.
filtered_products=df.groupby("Region").filter(lambda x: len(x)>1)
print(filtered_products)

  Region Product  Sales  Discount  normalized_sales  Discount_Diff
0  North       A    100         5          0.666667              5
1  North       B    200        10          1.333333              0
2  South       A    300        15          0.857143              5
3  South       C    400        20          1.142857              0
4   East       B    150         5          0.750000              5
5   East       C    250        10          1.250000              0
6   West       A    500        25          0.909091              5
7   West       B    600        30          1.090909              0


# Apply

In [ ]:
# For each region, return the top 1 product by sales.
top_sales=df.groupby("Region").apply(lambda x:x.nlargest(1,"Sales"))
print(top_sales)

         Product  Sales  Discount  normalized_sales  Discount_Diff
Region                                                            
East   5       C    250        10          1.250000              0
North  1       B    200        10          1.333333              0
South  3       C    400        20          1.142857              0
West   7       B    600        30          1.090909              0


In [19]:
# For each region, return the DF sorted by discount descending.
sorted_discount=df.groupby("Region").apply(lambda x: x.sort_values("Discount",ascending=False))
print(sorted_discount)

         Product  Sales  Discount  normalized_sales  Discount_Diff
Region                                                            
East   5       C    250        10          1.250000              0
       4       B    150         5          0.750000              5
North  1       B    200        10          1.333333              0
       0       A    100         5          0.666667              5
South  3       C    400        20          1.142857              0
       2       A    300        15          0.857143              5
West   7       B    600        30          1.090909              0
       6       A    500        25          0.909091              5


# Custom Aggregation

In [21]:
# For each region, calculate: total sales, average discount and sales range(max - min)
sales_data=df.groupby("Region")["Sales"].agg(total_sales="sum",average_discount="mean",range=lambda x: x.max()-x.min())
print(sales_data)

        total_sales  average_discount  range
Region                                      
East            400             200.0    100
North           300             150.0    100
South           700             350.0    100
West           1100             550.0    100


In [22]:
# For each product, calculate both mean and sum of sales.
product_data=df.groupby("Product")["Sales"].agg(avg_sales="mean",total_sales="sum")
print(product_data)

          avg_sales  total_sales
Product                         
A        300.000000          900
B        316.666667          950
C        325.000000          650


# Size vs Count

In [23]:
# Count how many rows per region.
rows_cout=df.groupby("Region").size()
print(rows_cout)

Region
East     2
North    2
South    2
West     2
dtype: int64


In [24]:
# Count how many non-NaN sales entries per region.
sales_entries_count=df.groupby("Region")["Sales"].count()
print(sales_entries_count)

Region
East     2
North    2
South    2
West     2
Name: Sales, dtype: int64


In [ ]:
# Group by region and sum sales, but keep the original order of regions.
print(df.groupby(["Region"],sort=False)["Sales"].sum())

Region
North     300
South     700
East      400
West     1100
Name: Sales, dtype: int64


# Observed=True

In [ ]:
df = pd.DataFrame({
    "Region": pd.Categorical(["North","South","East","West"], 
                             categories=["North","South","East","West","Central"]),
    "Sales": [100,200,300,400]
})

print(df)